# Numeric relationships with delivery duration

This notebook examines rank-based relationships between `Delivery_Time` and selected numeric features, with clear plots for the most relevant pairs. It reads the approved cleaned CSV from Downloads and saves figures in `outputs/`; it does not change the source file.

`Delivery_Time` is treated as delivery duration and its unit is left unspecified. `Pickup_Delay_Minutes` is order-to-pickup duration in minutes. The dataset has no promised delivery time, actual delivery timestamp, or delivery status; this notebook does not calculate on-time, late, completion, or promise-based delay rates.

Distance is calculated as **estimated straight-line distance** in kilometers using the great-circle formula. Rows with missing store coordinates are excluded from distance analysis, as are rows with missing or invalid drop coordinates. It is not road distance. Correlations describe association, not causation.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

csv_path = Path.home() / "Downloads" / "amazon_delivery_cleaned_updated.csv"
repo_root = Path.cwd().parent if Path.cwd().name == "Notebooks" else Path.cwd()
output_dir = repo_root / "outputs"
output_dir.mkdir(exist_ok=True)
df = pd.read_csv(csv_path)

assert df.shape == (43_739, 28), f"Unexpected approved-file shape: {df.shape}"
assert df["Delivery_Time"].notna().all(), "Delivery_Time has missing values; review before proceeding"
print(f"Source: {csv_path}")
print(f"Dataset: {df.shape[0]:,} rows, {df.shape[1]} columns")

## Estimated straight-line distance

The coordinate calculation uses only records with all four geographic coordinates present and within latitude and longitude bounds. In particular, no record with missing store coordinates is included. Great-circle distance is calculated in kilometers and remains an estimate of straight-line distance, not road distance or route length.

In [ ]:
coordinate_fields = ["Store_Latitude", "Store_Longitude", "Drop_Latitude", "Drop_Longitude"]
coordinates = df[coordinate_fields].apply(pd.to_numeric, errors="coerce")
valid_coordinates = (
    coordinates["Store_Latitude"].between(-90, 90)
    & coordinates["Drop_Latitude"].between(-90, 90)
    & coordinates["Store_Longitude"].between(-180, 180)
    & coordinates["Drop_Longitude"].between(-180, 180)
    & coordinates["Store_Latitude"].notna()
    & coordinates["Store_Longitude"].notna()
    & coordinates["Drop_Latitude"].notna()
    & coordinates["Drop_Longitude"].notna()
)
distance_km = pd.Series(np.nan, index=df.index, name="Estimated_Straight_Line_Distance_Km")
lat1 = np.radians(coordinates.loc[valid_coordinates, "Store_Latitude"])
lat2 = np.radians(coordinates.loc[valid_coordinates, "Drop_Latitude"])
dlat = lat2 - lat1
dlon = np.radians(
    coordinates.loc[valid_coordinates, "Drop_Longitude"]
    - coordinates.loc[valid_coordinates, "Store_Longitude"]
)
a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
distance_km.loc[valid_coordinates] = 6371.0 * 2 * np.arcsin(np.sqrt(a.clip(0, 1)))
df["Estimated_Straight_Line_Distance_Km"] = distance_km

print(f"Rows with missing store coordinates excluded: {int(coordinates[["Store_Latitude", "Store_Longitude"]].isna().any(axis=1).sum()):,}")
print(f"Rows eligible for estimated straight-line distance analysis: {int(valid_coordinates.sum()):,}")
print(f"Estimated distances above 50 km retained: {int(distance_km.gt(50).sum()):,}")

## Spearman correlation analysis

Spearman's rank correlation is used because it measures monotonic association and is less sensitive than Pearson correlation to extreme values. Raw latitude and longitude are not treated as meaningful numeric predictors. Pairwise sample sizes are shown alongside correlations because missing order-time, pickup, and coordinate fields reduce the usable sample for some pairs.

In [ ]:
numeric_fields = [
    "Delivery_Time",
    "Agent_Age",
    "Agent_Rating",
    "Pickup_Delay_Minutes",
    "Order_Hour",
    "Order_DayOfWeek",
    "Estimated_Straight_Line_Distance_Km",
]
numeric_data = df[numeric_fields]
spearman = numeric_data.corr(method="spearman")
pair_counts = numeric_data.notna().astype(int).T @ numeric_data.notna().astype(int)
correlation_with_delivery = pd.DataFrame({
    "feature": [field for field in numeric_fields if field != "Delivery_Time"],
    "spearman_rho": [spearman.loc["Delivery_Time", field] for field in numeric_fields if field != "Delivery_Time"],
    "pair_records": [int(pair_counts.loc["Delivery_Time", field]) for field in numeric_fields if field != "Delivery_Time"],
}).sort_values("spearman_rho", key=lambda values: values.abs(), ascending=False)
display(correlation_with_delivery.round({"spearman_rho": 3}))

heatmap_annotations = spearman.copy().astype(object)
for row in spearman.index:
    for column in spearman.columns:
        heatmap_annotations.loc[row, column] = f"{spearman.loc[row, column]:.2f}\nn={int(pair_counts.loc[row, column]):,}"

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(spearman, dtype=bool), k=1)
heatmap_labels = ["Delivery_Time", "Agent_Age", "Agent_Rating", "Pickup_Delay\n(minutes)", "Order_Hour", "Order_DayOfWeek", "Est. straight-line\ndistance (km)"]
sns.heatmap(
    spearman, mask=mask, annot=heatmap_annotations, fmt="", annot_kws={"fontsize": 8}, cmap="vlag", center=0,
    vmin=-1, vmax=1, square=True, linewidths=.5, xticklabels=heatmap_labels, yticklabels=heatmap_labels,
    cbar_kws={"label": "Spearman rank correlation (rho)"}, ax=ax,
)
ax.tick_params(axis="x", labelrotation=35)
ax.tick_params(axis="y", labelrotation=0)
ax.set_title("Rank correlations among delivery duration and numeric features")
fig.text(.5, .01, "Cells show Spearman rho and pairwise n; raw coordinates excluded. Correlation indicates association, not cause.", ha="center", fontsize=9)
fig.tight_layout(rect=(0, .04, 1, 1))
fig.savefig(output_dir / "numeric_relationship_correlation_heatmap.png", dpi=180, bbox_inches="tight")
plt.show()

## Delivery duration and selected numeric features

Each panel shows a reproducible random sample of up to 6,000 complete observations to reduce overplotting. The line connects median delivery duration across ordered feature bins. This visual summary does not adjust for other features. The `Delivery_Time` unit remains unspecified.

In [ ]:
relationship_fields = ["Agent_Age", "Agent_Rating", "Pickup_Delay_Minutes", "Order_Hour"]
titles = {
    "Agent_Age": "Agent age",
    "Agent_Rating": "Agent rating",
    "Pickup_Delay_Minutes": "Order-to-pickup duration (minutes)",
    "Order_Hour": "Order hour (as recorded)",
}
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
rng = np.random.default_rng(2026)
for ax, field in zip(axes.flat, relationship_fields):
    sample = df[[field, "Delivery_Time"]].dropna()
    if len(sample) > 6_000:
        sample = sample.iloc[np.sort(rng.choice(len(sample), size=6_000, replace=False))]
    ax.scatter(sample[field], sample["Delivery_Time"], s=9, alpha=.18, color="#31688e", edgecolors="none")
    if field == "Pickup_Delay_Minutes":
        grouped = df.groupby(field, observed=True)["Delivery_Time"].median().dropna()
    else:
        bins = pd.qcut(df[field], q=12, duplicates="drop")
        grouped = df.groupby(bins, observed=True)["Delivery_Time"].median().dropna()
        grouped.index = [interval.mid for interval in grouped.index]
    ax.plot(grouped.index.astype(float), grouped.values, color="#d1495b", linewidth=2.5, marker="o", markersize=4, label="Binned median")
    rho = spearman.loc["Delivery_Time", field]
    n = int(pair_counts.loc["Delivery_Time", field])
    ax.set_title(
        f"Delivery duration vs. {titles[field]}\nSpearman rho={rho:.2f} | pair n={n:,} | plotted n={len(sample):,}",
        fontsize=9,
    )
    ax.set(xlabel=titles[field], ylabel="Delivery_Time (unit unspecified)")
    ax.legend(frameon=False, fontsize=8)
fig.suptitle("Relationships between delivery duration and numeric features", fontsize=15, y=1.02)
fig.text(.5, .005, "Points are a deterministic sample; red markers show medians by ordered bins. Associations are descriptive.", ha="center", fontsize=9)
fig.tight_layout()
fig.savefig(output_dir / "delivery_duration_numeric_relationships.png", dpi=180, bbox_inches="tight")
plt.show()

## Delivery duration and estimated straight-line distance

This plot includes only records with valid store and drop coordinates, so records with missing store coordinates are excluded. A hexbin view shows where observations concentrate; a binned median and interquartile range summarizes delivery duration across ordered distance bins. The horizontal axis uses a logarithmic scale so typical local distances remain visible alongside retained extreme estimates.

In [ ]:
distance_data = df[["Estimated_Straight_Line_Distance_Km", "Delivery_Time"]].dropna()
distance_data = distance_data.loc[distance_data["Estimated_Straight_Line_Distance_Km"].gt(0)]
distance_edges = np.geomspace(distance_data["Estimated_Straight_Line_Distance_Km"].min(), distance_data["Estimated_Straight_Line_Distance_Km"].max(), 21)
distance_bins = pd.cut(distance_data["Estimated_Straight_Line_Distance_Km"], bins=distance_edges, include_lowest=True)
distance_summary = distance_data.groupby(distance_bins, observed=True)["Delivery_Time"].agg(
    records="count", median="median", q1=lambda values: values.quantile(.25), q3=lambda values: values.quantile(.75)
)
distance_summary["distance_geometric_midpoint_km"] = [np.sqrt(interval.left * interval.right) for interval in distance_summary.index]
display(distance_summary.round(2))

fig, ax = plt.subplots(figsize=(10, 6))
hb = ax.hexbin(
    distance_data["Estimated_Straight_Line_Distance_Km"], distance_data["Delivery_Time"],
    gridsize=45, mincnt=1, xscale="log", cmap="Blues", bins="log", linewidths=0,
)
ax.errorbar(
    distance_summary["distance_geometric_midpoint_km"], distance_summary["median"],
    yerr=[distance_summary["median"] - distance_summary["q1"], distance_summary["q3"] - distance_summary["median"]],
    color="#d1495b", marker="o", markersize=4, linestyle="none", capsize=2, label="Log-binned median and IQR",
)
fig.colorbar(hb, ax=ax, label="Records per hexagon (log scale)")
ax.set(title=f"Delivery duration by estimated straight-line distance (n={len(distance_data):,})", xlabel="Estimated straight-line distance (km, log scale)", ylabel="Delivery_Time (unit unspecified)")
ax.legend(frameon=False)
fig.text(.5, .01, "Valid coordinate pairs only; missing store coordinates excluded. Estimated straight-line distance is not road distance.", ha="center", fontsize=9)
fig.tight_layout(rect=(0, .04, 1, 1))
fig.savefig(output_dir / "delivery_duration_estimated_distance_relationship.png", dpi=180, bbox_inches="tight")
plt.show()

distance_rho = float(spearman.loc["Delivery_Time", "Estimated_Straight_Line_Distance_Km"])
print(f"Spearman correlation for estimated straight-line distance and Delivery_Time: rho={distance_rho:.3f}, n={len(distance_data):,}")

## Interpretation and limitations

- Spearman correlation summarizes monotonic association, not cause and effect. These are unadjusted, exploratory comparisons.
- `Delivery_Time` is delivery duration with unit unspecified; the plots preserve that label.
- `Pickup_Delay_Minutes` is the order-to-pickup duration in minutes, not a delivery delay against a promise.
- Distance is estimated straight-line distance. Rows with missing store coordinates are excluded from that analysis, and the figure reports the valid pair count. Road network, route, and traffic path are not represented.
- The data has no promised delivery time, actual delivery timestamp, or delivery status; no on-time, late, completion, or promise-based delay rates are calculated.
- Correlation values may be influenced by feature distributions, missingness, outliers, and unadjusted group differences. They do not establish causality.